# Ch 13　Functional API：用裝飾器寫 graph 的另一條路

> **本 notebook 完全不需要 API key。**

In [ ]:
!uv pip install -q langgraph

## 13.2　@task + @entrypoint：寫文章，然後暫停請人審核

In [ ]:
from langgraph.func import entrypoint, task
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

@task
def write_essay(topic: str) -> str:
    """寫一篇文章（假裝是個昂貴的長任務）。"""
    print(f"  ✍️ 正在寫關於「{topic}」的文章…（這行待會驗證『不會重跑』）")
    return f"一篇關於「{topic}」的文章"

@entrypoint(checkpointer=InMemorySaver())
def workflow(topic: str) -> dict:
    essay = write_essay(topic).result()     # @task 回傳 future-like，要 .result() 取值
    ok = interrupt({"essay": essay, "action": "核准 / 退回？"})
    return {"essay": essay, "is_approved": ok}

config = {"configurable": {"thread_id": "essay-1"}}
print("第一次跑（寫文章 → 暫停）：")
for item in workflow.stream("貓", config):
    print("  ", item)

## 13.3　關鍵：@task 結果存進 checkpoint，resume 不重算

注意：resume 後，上面那行「✍️ 正在寫…」**不會再印一次**——因為 task 結果是從 checkpoint 載回的。

In [ ]:
print("回復（核准）：")
for item in workflow.stream(Command(resume=True), config):
    print("  ", item)
print("=> 上面沒有再出現『✍️ 正在寫…』，證明 write_essay 沒重跑（呼應 Ch9：副作用包進 task）。")